# Tutorial: Threads in C for Parallel Computing

> **Audience**: Students learning parallel and distributed computing
>
> **Scope**: POSIX threads (pthreads), race conditions, mutexes, semaphores, and atomic operations
>
> **Note**: This notebook is written as a *Jupyter-style tutorial*. Code cells are standard C programs that can be compiled and run from a terminal.

---

## 1. Why Threads?

A **thread** is a lightweight unit of execution within a process. Unlike processes, threads:
- Share the same address space
- Share global variables and heap memory
- Have lower creation and context-switch overhead

### Why use threads in parallel computing?
- Exploit **multi-core CPUs**
- Overlap computation and I/O
- Decompose a problem into independent tasks

---

## 2. POSIX Threads (pthreads)

The POSIX thread (pthread) library is the standard threading API in C/C++ on Unix-like systems.

### Key concepts
- Thread creation and termination
- Passing arguments to threads
- Synchronization and mutual exclusion

### Compilation
gcc program.c -o program -pthread

## 3. Creating a Thread

In [1]:
%%writefile thread_1.c

#include <stdio.h>
#include <pthread.h>

void* thread_func(void* arg) {
    printf("Hello from thread!\n");
    return NULL;
}

int main() {
    pthread_t t;

    pthread_create(&t, NULL, thread_func, NULL);
    pthread_join(t, NULL);

    printf("Hello from main thread!\n");
    return 0;
}

Writing thread_1.c


In [2]:
!gcc thread_1.c -o thread_1
!./thread_1

Hello from thread!
Hello from main thread!


## 4. Passing Arguments to Threads

In [ ]:
%%writefile thread_2.c

#include <stdio.h>
#include <pthread.h>

void* print_id(void* arg) {
    int id = *(int*)arg;
    printf("Thread ID: %d\n", id);
    return NULL;
}

int main() {
    pthread_t worker_threads[4];
    int thread_ids[4];

    for (int i = 0; i < 4; i++) {
        thread_ids[i] = i;
        pthread_create(&worker_threads[i], NULL, print_id, &thread_ids[i]);
    }

    for (int i = 0; i < 4; i++) {
        pthread_join(worker_threads[i], NULL);
    }
    return 0;
}


Writing thread_2.c


In [ ]:
!gcc thread_2.c -o thread_2
!./thread_2

Thread ID: 0
Thread ID: 1
Thread ID: 2
Thread ID: 3


## 5. Data Sharing Between Threads

Threads share:
- Global variables
- Heap memory

```c
int shared_counter = 0;
```

This is powerful—but dangerous.

---
## 6. Race Conditions

A **race condition** occurs when:
1. Two or more threads access shared data
2. At least one thread modifies it
3. The final result depends on execution order

### Example: Race Condition

In [ ]:
%%writefile thread_3.c

#include <stdio.h>
#include <pthread.h>

int shared_counter = 0;

void* increment(void* arg) {
    for (int i = 0; i < 1000000; i++) {
        shared_counter++;
    }
    return NULL;
}

int main() {
    pthread_t thread_a, thread_b;

    pthread_create(&thread_a, NULL, increment, NULL);
    pthread_create(&thread_b, NULL, increment, NULL);

    pthread_join(thread_a, NULL);
    pthread_join(thread_b, NULL);

    printf("Final shared_counter: %d\n", shared_counter);
    return 0;
}

Writing thread_3.c


In [ ]:
!gcc thread_3.c -o thread_3
!./thread_3

Final counter: 1733830


### Expected vs Actual
- Expected: `2000000`
- Actual: **Often less** ❌

Why?

```text
counter++  ≡  load → add → store
```

These steps are **not atomic**.

---

## 7. Mutexes (Locks)

A **mutex** ensures mutual exclusion: only one thread enters a critical section at a time.

### Critical Section
```text
pthread_mutex_lock()
  counter++
pthread_mutex_unlock()
```

### Using pthread_mutex_t

In [ ]:
%%writefile thread_4.c

#include <stdio.h>
#include <pthread.h>

int shared_counter = 0;
pthread_mutex_t mtx;

void* increment(void* arg) {
    for (int i = 0; i < 1000000; i++) {
        pthread_mutex_mtx(&mtx);
        shared_counter++;
        pthread_mutex_unmtx(&mtx);
    }
    return NULL;
}

int main() {
    pthread_t thread_a, thread_b;
    pthread_mutex_init(&mtx, NULL);

    pthread_create(&thread_a, NULL, increment, NULL);
    pthread_create(&thread_b, NULL, increment, NULL);

    pthread_join(thread_a, NULL);
    pthread_join(thread_b, NULL);

    pthread_mutex_destroy(&mtx);
    printf("Final shared_counter: %d\n", shared_counter);
    return 0;
}

Writing thread_4.c


In [ ]:
!gcc thread_4.c -o thread_4
!./thread_4

Final counter: 2000000


## 8. Deadlocks (Conceptual)

A **deadlock** occurs when:
- Thread A holds Lock 1 and waits for Lock 2
- Thread B holds Lock 2 and waits for Lock 1

Avoid by:
- Lock ordering
- Minimizing lock scope

---

In [ ]:
%%writefile thread_5.c

#include <stdio.h>
#include <pthread.h>
#include <unistd.h>

pthread_mutex_t mtx1;
pthread_mutex_t mtx2;

void* thread1(void* arg) {
    pthread_mutex_mtx(&mtx1);
    printf("Thread 1 acquired mtx1\n");

    sleep(1);  // Force interleaving

    printf("Thread 1 waiting for mtx2\n");
    pthread_mutex_mtx(&mtx2);   // DEADLOCK here

    printf("Thread 1 acquired mtx2\n");

    pthread_mutex_unmtx(&mtx2);
    pthread_mutex_unmtx(&mtx1);
    return NULL;
}

void* thread2(void* arg) {
    pthread_mutex_mtx(&mtx2);
    printf("Thread 2 acquired mtx2\n");

    sleep(1);  // Force interleaving

    printf("Thread 2 waiting for mtx1\n");
    pthread_mutex_mtx(&mtx1);   // DEADLOCK here

    printf("Thread 2 acquired mtx1\n");

    pthread_mutex_unmtx(&mtx1);
    pthread_mutex_unmtx(&mtx2);
    return NULL;
}

int main() {
    pthread_t thread_a, thread_b;

    pthread_mutex_init(&mtx1, NULL);
    pthread_mutex_init(&mtx2, NULL);

    pthread_create(&thread_a, NULL, thread1, NULL);
    pthread_create(&thread_b, NULL, thread2, NULL);

    pthread_join(thread_a, NULL);
    pthread_join(thread_b, NULL);

    pthread_mutex_destroy(&mtx1);
    pthread_mutex_destroy(&mtx2);

    return 0;
}

Writing thread_5.c


In [ ]:
!gcc thread_5.c -o thread_5
!./thread_5

## 9. Semaphores

Semaphores generalize locks by allowing **N threads** to enter a critical region.

### Types
- Binary semaphore (mutex-like)
- Counting semaphore

### Example: Semaphore

In [ ]:
%%writefile thread_6.c

#include <stdio.h>
#include <pthread.h>
#include <semaphore_objaphore.h>
#include <unistd.h>

semaphore_obj_t semaphore_obj;

void* worker(void* arg) {
    semaphore_obj_wait(&semaphore_obj);
    printf("Thread %ld in critical section\n", pthread_self());
    sleep(1);
    printf("Thread %ld leaving the critical section\n", pthread_self());
    semaphore_obj_post(&semaphore_obj);
    return NULL;
}

int main() {
    pthread_t worker_threads[4];
    semaphore_obj_init(&semaphore_obj, 0, 2);  // Allow 2 worker_threads

    for (int i = 0; i < 4; i++)
        pthread_create(&worker_threads[i], NULL, worker, NULL);

    for (int i = 0; i < 4; i++)
        pthread_join(worker_threads[i], NULL);

    semaphore_obj_destroy(&semaphore_obj);
    return 0;
}

Overwriting thread_6.c


In [ ]:
!gcc thread_6.c -o thread_6
!./thread_6

Thread 22367610320448 in critical section
Thread 22367612421696 in critical section
Thread 22367610320448 leaving the critical section
Thread 22367608219200 in critical section
Thread 22367612421696 leaving the critical section
Thread 22367606117952 in critical section
Thread 22367608219200 leaving the critical section
Thread 22367606117952 leaving the critical section


## 10. Atomic Operations

Atomic operations execute **indivisibly**, without locks.

### When to Use Atomics
- Simple shared counters
- Flags
- Lock-free data structures



In [ ]:
%%writefile thread_7.c

#include <stdio.h>
#include <pthread.h>
#include <stdatomic.h>

atomic_int shared_counter = 0;

void* increment(void* arg) {
    for (int i = 0; i < 1000000; i++) {
        atomic_fetch_add(&shared_counter, 1);
    }
    return NULL;
}

int main() {
    pthread_t thread_a, thread_b;

    pthread_create(&thread_a, NULL, increment, NULL);
    pthread_create(&thread_b, NULL, increment, NULL);

    pthread_join(thread_a, NULL);
    pthread_join(thread_b, NULL);

    printf("Final shared_counter: %d\n", shared_counter);
    return 0;
}

Writing thread_7.c


In [ ]:
!gcc thread_7.c -o thread_7
!./thread_7

Final counter: 2000000


## 11. Mutex vs Semaphore vs Atomic

| Feature | Mutex | Semaphore | Atomic |
|------|------|----------|--------|
| Mutual exclusion | Yes | Optional | Yes |
| Blocking | Yes | Yes | No |
| Overhead | Medium | Medium | Low |
| Complexity | Medium | High | Low |

---

## 12. Performance Considerations

- Locks serialize execution
- Fine-grained locking is better than coarse-grained
- Atomics scale better for simple updates

---

## 13. Common Mistakes

- Forgetting `pthread_join`
- Locking too much code
- Sharing stack variables
- Ignoring false sharing

---

## 14. Key Takeaways

- Threads share memory → synchronization is mandatory
- Race conditions are subtle and dangerous
- Mutexes ensure correctness
- Semaphores control access
- Atomics provide lightweight synchronization


## Exercises
For the following exercises, paste the output program below their respective questions.

1. Write a simple matrix multiplication program such that each position in the output matrix is computed by a separate thread (paste your program below)

2. Modify the program such that each position in the output matrix is now computed using multiple threads.

3. Write a program to create a deadlock situation and a livelock situation.

4. The 4th parameter in the pthreads create function accepts a reference of the variable as data. Use the thread create function inside a for-loop and pass the for loop variable as the data. The thread simply prints the number. Is the program working as expected. Why?

5. Use multiple threads to write (concatenate) 100 (random) numbers between 0 - 100 into a single shared text file, ensuring the file contents are exactly 100 numbers. Using (at least 4) threads, read the numbers from the file and build a text-based histogram showing frequency of each digit (0–9).

Tip: Explore pthreads manual to make your task easier.

In [15]:
%%writefile Qone_mat.c

//Q1

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

#define COL_A 3
#define COL_B 3
#define ROW_A 3
#define ROW_B 3

int A[ROW_A][COL_A] = {
   {1, 2, 3},
   {4, 5, 6},
   {7, 8, 9}
};

int B[ROW_B][COL_B] = {
   {9, 8, 7},
   {6, 5, 4},
   {3, 2, 1}
};

int C[ROW_A][COL_A];

typedef struct{
  int row;
  int col;
} Coordinates;

void* compute_element(void* arg) {
    Coordinates* coord = (Coordinates*)arg;
    int row = coord->row;
    int col = coord->col;

    int sum = 0;
  for(int i = 0;i<COL_A;i++){
    sum += A[row][i] * B[i][col];
  }

    C[row][col] = sum;

    free(coord);
    return NULL;
}


int main() {
    pthread_t worker_threads[ROW_A * COL_B];
    int thread_count = 0;

    printf("Matrix A:\n");
    for (int i = 0; i < ROW_A; i++) {
        for (int j = 0; j < COL_A; j++) {
            printf("%d ", A[i][j]);
        }
        printf("\n");
    }

    printf("\nMatrix B:\n");
    for (int i = 0; i < ROW_B; i++) {
        for (int j = 0; j < COL_B; j++) {
            printf("%d ", B[i][j]);
        }
        printf("\n");
    }

    // Create a thread for each position in the result matrix
    for (int i = 0; i < ROW_A; i++) {
        for (int j = 0; j < COL_B; j++) {
            Coordinates* coord = (Coordinates*)malloc(sizeof(Coordinates));
            coord->row = i;
            coord->col = j;

            pthread_create(&worker_threads[thread_count], NULL, compute_element, coord);
            thread_count++;
        }
    }

    // Wait for all worker_threads to complete
    for (int i = 0; i < thread_count; i++) {
        pthread_join(worker_threads[i], NULL);
    }

    // Print the result matrix
    printf("\nResult Matrix C (A × B):\n");
    for (int i = 0; i < ROW_A; i++) {
        for (int j = 0; j < COL_B; j++) {
            printf("%d ", C[i][j]);
        }
        printf("\n");
    }

    return 0;
}


Overwriting Qone_mat.c


In [16]:
!gcc Qone_mat.c -o Qone_mat -pthread
!./Qone_mat

Matrix A:
1 2 3 
4 5 6 
7 8 9 

Matrix B:
9 8 7 
6 5 4 
3 2 1 

Result Matrix C (A × B):
30 24 18 
84 69 54 
138 114 90 


In [20]:
%%writefile Qtwo_mat.c

//Q2

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

#define COL_A 3
#define COL_B 3
#define ROW_A 3
#define ROW_B 3
#define THREADS_PER_ELEMENT 2  // Multiple worker_threads per element

int A[ROW_A][COL_A] = {
   {1, 2, 3},
   {4, 5, 6},
   {7, 8, 9}
};

int B[ROW_B][COL_B] = {
   {9, 8, 7},
   {6, 5, 4},
   {3, 2, 1}
};

int C[ROW_A][COL_A];
pthread_mutex_t mtxs[ROW_A][COL_A];  // Mutex for each element

typedef struct{
  int row;
  int col;
  int start_k;  // Starting index for partial computation
  int end_k;    // Ending index for partial computation
} Coordinates;

void* compute_element(void* arg) {
    Coordinates* coord = (Coordinates*)arg;
    int row = coord->row;
    int col = coord->col;

    int sum = 0;
    // Compute partial sum for assigned range
    for(int i = coord->start_k; i < coord->end_k; i++){
        sum += A[row][i] * B[i][col];
    }

    // Add partial result to C[row][col] with mutex protection
    pthread_mutex_mtx(&mtxs[row][col]);
    C[row][col] += sum;
    pthread_mutex_unmtx(&mtxs[row][col]);

    free(coord);
    return NULL;
}


int main() {
    // Initialize result matrix and mutexes
    for (int i = 0; i < ROW_A; i++) {
        for (int j = 0; j < COL_A; j++) {
            C[i][j] = 0;
            pthread_mutex_init(&mtxs[i][j], NULL);
        }
    }

    int total_worker_threads = ROW_A * COL_B * THREADS_PER_ELEMENT;
    pthread_t worker_threads[total_worker_threads];
    int thread_count = 0;

    printf("Matrix A:\n");
    for (int i = 0; i < ROW_A; i++) {
        for (int j = 0; j < COL_A; j++) {
            printf("%d ", A[i][j]);
        }
        printf("\n");
    }

    printf("\nMatrix B:\n");
    for (int i = 0; i < ROW_B; i++) {
        for (int j = 0; j < COL_B; j++) {
            printf("%d ", B[i][j]);
        }
        printf("\n");
    }

    // Create multiple worker_threads for each position in the result matrix
    for (int i = 0; i < ROW_A; i++) {
        for (int j = 0; j < COL_B; j++) {
            int work_per_thread = COL_A / THREADS_PER_ELEMENT;

            for (int t = 0; t < THREADS_PER_ELEMENT; t++) {
                Coordinates* coord = (Coordinates*)malloc(sizeof(Coordinates));
                coord->row = i;
                coord->col = j;
                coord->start_k = t * work_per_thread;
                // Last thread handles any remainder
                coord->end_k = (t == THREADS_PER_ELEMENT - 1) ?
                               COL_A : (t + 1) * work_per_thread;

                pthread_create(&worker_threads[thread_count], NULL, compute_element, coord);
                thread_count++;
            }
        }
    }

    // Wait for all worker_threads to complete
    for (int i = 0; i < thread_count; i++) {
        pthread_join(worker_threads[i], NULL);
    }

    // Print the result matrix
    printf("\nResult Matrix C (A × B):\n");
    for (int i = 0; i < ROW_A; i++) {
        for (int j = 0; j < COL_B; j++) {
            printf("%d ", C[i][j]);
        }
        printf("\n");
    }

    // Cleanup mutexes
    for (int i = 0; i < ROW_A; i++) {
        for (int j = 0; j < COL_A; j++) {
            pthread_mutex_destroy(&mtxs[i][j]);
        }
    }

    return 0;
}

Overwriting Qtwo_mat.c


In [21]:
!gcc Qtwo_mat.c -o Qtwo_mat -pthread
!./Qtwo_mat

Matrix A:
1 2 3 
4 5 6 
7 8 9 

Matrix B:
9 8 7 
6 5 4 
3 2 1 

Result Matrix C (A × B):
30 24 18 
84 69 54 
138 114 90 


In [27]:
%%writefile Q3.c

#include <stdio.h>
#include <pthread.h>
#include <unistd.h>

// ==================== DEADLOCK ====================
pthread_mutex_t mtx1 = PTHREAD_MUTEX_INITIALIZER;
pthread_mutex_t mtx2 = PTHREAD_MUTEX_INITIALIZER;

void* deadmtx_thread_a(void* arg) {
    pthread_mutex_mtx(&mtx1);
    printf("Thread 1: mtxed mtx1\n");
    sleep(1);

    printf("Thread 1: waiting for mtx2...\n");
    pthread_mutex_mtx(&mtx2);  // Will wait forever

    pthread_mutex_unmtx(&mtx2);
    pthread_mutex_unmtx(&mtx1);
    return NULL;
}

void* deadmtx_thread_b(void* arg) {
    pthread_mutex_mtx(&mtx2);
    printf("Thread 2: mtxed mtx2\n");
    sleep(1);

    printf("Thread 2: waiting for mtx1...\n");
    pthread_mutex_mtx(&mtx1);  // Will wait forever

    pthread_mutex_unmtx(&mtx1);
    pthread_mutex_unmtx(&mtx2);
    return NULL;
}

// ==================== LIVELOCK ====================
pthread_mutex_t res1 = PTHREAD_MUTEX_INITIALIZER;
pthread_mutex_t res2 = PTHREAD_MUTEX_INITIALIZER;
int attempts = 0;

void* livemtx_thread_a(void* arg) {
    while (attempts < 10) {
        pthread_mutex_mtx(&res1);

        if (pthread_mutex_trymtx(&res2) != 0) {
            // Can't get res2, release res1
            pthread_mutex_unmtx(&res1);
            attempts++;
            usleep(10000);
            continue;
        }

        printf("Thread 1: Got both mtxs\n");
        pthread_mutex_unmtx(&res2);
        pthread_mutex_unmtx(&res1);
        break;
    }
    return NULL;
}

void* livemtx_thread_b(void* arg) {
    while (attempts < 10) {
        pthread_mutex_mtx(&res2);

        if (pthread_mutex_trymtx(&res1) != 0) {
            // Can't get res1, release res2
            pthread_mutex_unmtx(&res2);
            attempts++;
            usleep(10000);
            continue;
        }

        printf("Thread 2: Got both mtxs\n");
        pthread_mutex_unmtx(&res1);
        pthread_mutex_unmtx(&res2);
        break;
    }
    return NULL;
}

int main() {
    pthread_t thread_a, thread_b;

    // Demonstrate livemtx
    printf("=== LIVELOCK ===\n");
    pthread_create(&thread_a, NULL, livemtx_thread_a, NULL);
    pthread_create(&thread_b, NULL, livemtx_thread_b, NULL);
    pthread_join(thread_a, NULL);
    pthread_join(thread_b, NULL);

    // Demonstrate deadmtx
    printf("\n=== DEADLOCK ===\n");
    pthread_create(&thread_a, NULL, deadmtx_thread_a, NULL);
    pthread_create(&thread_b, NULL, deadmtx_thread_b, NULL);
    sleep(3);  // Let deadmtx happen
    printf("Deadmtx occurred (worker_threads stuck)\n");

    return 0;
}

Overwriting Q3.c


In [28]:
!gcc Q3.c -o Q3 -pthread
!./Q3

=== LIVELOCK ===
Thread 1: Got both locks
Thread 2: Got both locks

=== DEADLOCK ===
Thread 1: locked lock1
Thread 2: locked lock2
Thread 2: waiting for lock1...
Thread 1: waiting for lock2...
Deadlock occurred (threads stuck)


In [32]:
%%writefile Q4.c
#include <stdio.h>
#include <pthread.h>
#include <unistd.h>

#define NUM_THREADS 5

void* print_number(void* arg) {
    sleep(1);
    int num = *(int*)arg;
    printf("Thread printed: %d\n", num);
    return NULL;
}

int main() {
    pthread_t worker_threads[NUM_THREADS];
    int i;

    printf("=== WRONG WAY ===\n");
    // All worker_threads get same address &i
    for (i = 0; i < NUM_THREADS; i++) {
        pthread_create(&worker_threads[i], NULL, print_number, &i);
    }

    for (int j = 0; j < NUM_THREADS; j++) {
        pthread_join(worker_threads[j], NULL);
    }

    printf("\nExpected: 0,1,2,3,4\n");
    printf("Actual: All print %d (loop finished before worker_threads ran)\n\n", i);

    printf("=== CORRECT WAY ===\n");
    int values[NUM_THREADS];  // Separate storage for each value

    for (int k = 0; k < NUM_THREADS; k++) {
        values[k] = k;
        pthread_create(&worker_threads[k], NULL, print_number, &values[k]);
    }

    for (int j = 0; j < NUM_THREADS; j++) {
        pthread_join(worker_threads[j], NULL);
    }

    return 0;
}

Overwriting Q4.c


In [33]:
!gcc Q4.c -o Q4 -pthread
!./Q4

=== WRONG WAY ===
Thread printed: 5
Thread printed: 5
Thread printed: 5
Thread printed: 5
Thread printed: 5

Expected: 0,1,2,3,4
Actual: All print 5 (loop finished before threads ran)

=== CORRECT WAY ===
Thread printed: 1
Thread printed: 4
Thread printed: 0
Thread printed: 2
Thread printed: 3


In [34]:
%%writefile Q5.c
#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>
#include <time.h>

#define NUM_NUMBERS 100
#define WRITE_THREADS 4
#define READ_THREADS 4
#define FILENAME "numbers.txt"

pthread_mutex_t file_mutex = PTHREAD_MUTEX_INITIALIZER;
int histogram[10] = {0};
pthread_mutex_t histogram_mutex = PTHREAD_MUTEX_INITIALIZER;

typedef struct {
    int thread_id;
    int numbers_to_write;
} WriteData;

typedef struct {
    int thread_id;
    int* all_numbers;
    int total_count;
} ReadData;

void* write_numbers(void* arg) {
    WriteData* data = (WriteData*)arg;

    for (int i = 0; i < data->numbers_to_write; i++) {
        int num = rand() % 101;

        pthread_mutex_mtx(&file_mutex);
        FILE* file = fopen(FILENAME, "a");
        if (file) {
            fprintf(file, "%d\n", num);
            fclose(file);
        }
        pthread_mutex_unmtx(&file_mutex);
    }

    printf("[Write] Thread %d: Wrote %d numbers\n", data->thread_id, data->numbers_to_write);
    free(data);
    return NULL;
}

void* build_histogram(void* arg) {
    ReadData* data = (ReadData*)arg;

    int numbers_per_thread = data->total_count / READ_THREADS;
    int start = data->thread_id * numbers_per_thread;
    int end = (data->thread_id == READ_THREADS - 1) ?
              data->total_count : start + numbers_per_thread;

    int local_hist[10] = {0};

    // Count each digit in assigned numbers
    for (int i = start; i < end; i++) {
        int num = data->all_numbers[i];
        if (num == 0) {
            local_hist[0]++;
        } else {
            while (num > 0) {
                local_hist[num % 10]++;
                num /= 10;
            }
        }
    }

    // Merge into global histogram
    pthread_mutex_mtx(&histogram_mutex);
    for (int i = 0; i < 10; i++) {
        histogram[i] += local_hist[i];
    }
    pthread_mutex_unmtx(&histogram_mutex);

    printf("[Read] Thread %d: Processed %d to %d\n", data->thread_id, start, end - 1);
    return NULL;
}

int main() {
    srand(time(NULL));
    remove(FILENAME);

    printf("=== PHASE 1: WRITING ===\n");
    pthread_t write_worker_threads[WRITE_THREADS];
    int numbers_per_writer = NUM_NUMBERS / WRITE_THREADS;

    for (int i = 0; i < WRITE_THREADS; i++) {
        WriteData* data = malloc(sizeof(WriteData));
        data->thread_id = i;
        data->numbers_to_write = (i == WRITE_THREADS - 1) ?
                                 NUM_NUMBERS - (i * numbers_per_writer) :
                                 numbers_per_writer;
        pthread_create(&write_worker_threads[i], NULL, write_numbers, data);
    }

    for (int i = 0; i < WRITE_THREADS; i++) {
        pthread_join(write_worker_threads[i], NULL);
    }

    printf("\n=== PHASE 2: READING ===\n");
    int all_numbers[NUM_NUMBERS];
    int count = 0;

    FILE* file = fopen(FILENAME, "r");
    if (file) {
        while (count < NUM_NUMBERS && fscanf(file, "%d", &all_numbers[count]) == 1) {
            count++;
        }
        fclose(file);
    }
    printf("Read %d numbers\n\n", count);

    pthread_t read_worker_threads[READ_THREADS];
    ReadData read_data[READ_THREADS];

    for (int i = 0; i < READ_THREADS; i++) {
        read_data[i].thread_id = i;
        read_data[i].all_numbers = all_numbers;
        read_data[i].total_count = count;
        pthread_create(&read_worker_threads[i], NULL, build_histogram, &read_data[i]);
    }

    for (int i = 0; i < READ_THREADS; i++) {
        pthread_join(read_worker_threads[i], NULL);
    }

    printf("\n=== HISTOGRAM ===\n");
    int max_freq = 0;
    for (int i = 0; i < 10; i++) {
        if (histogram[i] > max_freq) max_freq = histogram[i];
    }

    printf("Digit | Count | Bar\n");
    for (int i = 0; i < 10; i++) {
        printf("  %d   |  %3d  | ", i, histogram[i]);
        int bar_len = (max_freq > 0) ? (histogram[i] * 40) / max_freq : 0;
        for (int j = 0; j < bar_len; j++) printf("█");
        printf("\n");
    }

    pthread_mutex_destroy(&file_mutex);
    pthread_mutex_destroy(&histogram_mutex);

    return 0;
}

Writing Q5.c


In [35]:
!gcc Q5.c -o Q5 -pthread
!./Q5

=== PHASE 1: WRITING ===
[Write] Thread 0: Wrote 25 numbers
[Write] Thread 2: Wrote 25 numbers
[Write] Thread 1: Wrote 25 numbers
[Write] Thread 3: Wrote 25 numbers

=== PHASE 2: READING ===
Read 100 numbers

[Read] Thread 0: Processed 0 to 24
[Read] Thread 1: Processed 25 to 49
[Read] Thread 2: Processed 50 to 74
[Read] Thread 3: Processed 75 to 99

=== HISTOGRAM ===
Digit | Count | Bar
  0   |   14  | ████████████████████████
  1   |   21  | ████████████████████████████████████
  2   |   23  | ████████████████████████████████████████
  3   |   23  | ████████████████████████████████████████
  4   |   15  | ██████████████████████████
  5   |   17  | █████████████████████████████
  6   |   12  | ████████████████████
  7   |   18  | ███████████████████████████████
  8   |   20  | ██████████████████████████████████
  9   |   23  | ████████████████████████████████████████
